# 08 — Market Microstructure

**AFML Chapters 18–19** — López de Prado (2018)

Market microstructure reveals the mechanics of price formation:
who is trading, how much impact do they have, and how toxic is
the order flow?

This notebook demonstrates:
1. **VPIN** — Volume-synchronized Probability of Informed Trading
2. **Amihud illiquidity** — price impact per dollar volume
3. **Kyle's lambda** — regression-based price impact
4. **Roll spread** — implied bid-ask spread

---

In [ ]:
from _setup import *
from afml.microstructure import (
    vpin,
    amihud_illiquidity,
    kyle_lambda,
    roll_spread,
)

## 1. Load data

In [ ]:
panel = load_daily_bars()
btc = panel[panel["symbol"] == "BTC-USD"].sort_values("ts").drop_duplicates("ts", keep="last")
close = btc.set_index("ts")["close"]
volume = btc.set_index("ts")["volume"]

print(f"BTC: {len(close)} daily bars, {close.index.min().date()} to {close.index.max().date()}")

## 2. VPIN — Flow toxicity

In [ ]:
vpin_result = vpin(close, volume, n_buckets=100, window=20)
print(f"VPIN computed for {len(vpin_result)} volume buckets")
print(f"Mean VPIN: {vpin_result['vpin'].mean():.3f}")
print(f"Max  VPIN: {vpin_result['vpin'].max():.3f}")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(close, color=NAVY, lw=0.8)
ax.set_ylabel("Price (USD)")
ax.set_title("BTC price", fontweight="bold")

ax = axes[1]
ax.plot(vpin_result["bucket_end_date"], vpin_result["vpin"], color=TEAL, lw=0.8)
ax.fill_between(vpin_result["bucket_end_date"], 0, vpin_result["vpin"],
                alpha=0.2, color=TEAL)
mean_vpin = vpin_result["vpin"].mean()
ax.axhline(mean_vpin, color=RED, ls="--", lw=1, label=f"Mean: {mean_vpin:.3f}")
ax.set_ylabel("VPIN")
ax.set_title("VPIN — Probability of Informed Trading", fontweight="bold")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "08_vpin.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Liquidity measures: Amihud, Kyle, Roll

In [ ]:
amihud = amihud_illiquidity(close, volume, window=21)
kyle = kyle_lambda(close, volume, window=21)
roll = roll_spread(close, window=21)

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

ax = axes[0]
ax.plot(close, color=NAVY, lw=0.8)
ax.set_ylabel("Price")
ax.set_title("BTC price", fontweight="bold")

ax = axes[1]
ax.plot(amihud.dropna(), color=TEAL, lw=0.8)
ax.set_ylabel("Amihud λ")
ax.set_title("Amihud illiquidity (21d rolling)", fontweight="bold")
ax.set_yscale("log")

ax = axes[2]
ax.plot(kyle.dropna(), color=GOLD, lw=0.8)
ax.set_ylabel("Kyle λ")
ax.set_title("Kyle's lambda (21d rolling)", fontweight="bold")

ax = axes[3]
ax.plot(roll.dropna(), color=RED, lw=0.8)
ax.set_ylabel("Roll spread")
ax.set_title("Roll implied spread (21d rolling)", fontweight="bold")

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "08_liquidity.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Multi-asset liquidity comparison

In [ ]:
symbols = ["BTC-USD", "ETH-USD", "SOL-USD", "DOGE-USD", "XRP-USD"]
liq_results = []

for sym in symbols:
    asset = panel[panel["symbol"] == sym].sort_values("ts").drop_duplicates("ts", keep="last")
    c = asset.set_index("ts")["close"]
    v = asset.set_index("ts")["volume"]

    a = amihud_illiquidity(c, v, window=21).dropna()
    r = roll_spread(c, window=21).dropna()

    liq_results.append({
        "symbol": sym,
        "amihud_median": a.median(),
        "roll_spread_median": r.median(),
        "n_days": len(c),
    })

df_liq = pd.DataFrame(liq_results).set_index("symbol")
display(df_liq.style.format({
    "amihud_median": "{:.2e}",
    "roll_spread_median": "{:.2f}",
    "n_days": "{:,}",
}))

## 5. Summary

| Measure | What it captures | Crypto insight |
|---------|-----------------|----------------|
| VPIN | Flow toxicity (informed trading) | Spikes before large moves |
| Amihud | Price impact per dollar | BTC is most liquid; alts are less so |
| Kyle's λ | Regression-based price impact | Captures market depth |
| Roll spread | Implicit bid-ask spread | Wider during stress periods |

**For production:**
- Use VPIN as a real-time risk indicator (high VPIN → reduce exposure)
- Include Amihud/Roll as liquidity features in ML models
- Filter trades on illiquid assets using Amihud thresholds

---

*Next: [09_backtest_dangers.ipynb](09_backtest_dangers.ipynb) — Backtesting Dangers (AFML Ch 11)*